<a href="https://colab.research.google.com/github/Steco17/jsc-api-backend/blob/main/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JSC Translator - full training run

Fine-tunes NLLB-200-distilled-600M on every `data_ready` language-script target in `data/languages.json` using LoRA and selective language-token training.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

**One-time setup - GitHub token (repo is private):**
1. Create a fine-grained token at https://github.com/settings/tokens with read-only Contents access to `jsc-api-backend`.
2. In this notebook, click the key icon (Secrets) in the left sidebar.
3. Add a secret named `GITHUB_TOKEN` with that token as the value, and enable notebook access.

**Always use `Runtime > Run all`.** Do not skip the clone, dependency, preparation, validation, or smoke-test cells.

If a session disconnects, reconnect and run all cells again. Stable data fingerprints let the training cell resume the latest complete checkpoint from Google Drive. The initial full run is one epoch over roughly 3.37 million bidirectional rows; extend it only if evaluation supports another pass.

In [ ]:
!nvidia-smi

In [ ]:
import os

try:
    from google.colab import drive, userdata
except ModuleNotFoundError as exc:
    raise RuntimeError(
        'This notebook must run in Google Colab. Click the Open in Colab badge at the top.'
    ) from exc

drive.mount('/content/drive')

# Checkpoints and the final merged model live in Drive instead of the
# ephemeral Colab VM, so a runtime disconnect cannot erase the run.
OUT_DIR = '/content/drive/MyDrive/jsc_translator/model_out'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import base64
import os
import shutil
import subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
assert GITHUB_TOKEN, (
    'GITHUB_TOKEN is missing or not granted to this notebook. Open the Secrets panel, '
    'add the token, and enable notebook access.'
)

REPO = 'Steco17/jsc-api-backend'
REPO_URL = f'https://github.com/{REPO}.git'
REPO_DIR = '/content/jsc_api_backend'

# Pass authentication for this command only. The token is not written into
# .git/config, notebook output, or the repository's saved remote URL.
basic_auth = base64.b64encode(f'x-access-token:{GITHUB_TOKEN}'.encode()).decode()
auth_option = f'http.extraHeader=AUTHORIZATION: basic {basic_auth}'

if not os.path.isfile(f'{REPO_DIR}/requirements-train.txt'):
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ['git', '-c', auth_option, 'clone', REPO_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(
        ['git', '-C', REPO_DIR, '-c', auth_option, 'pull', '--ff-only'],
        check=True,
    )

assert os.path.isfile(f'{REPO_DIR}/scripts/finetune.py'), (
    'Repository setup is incomplete. Confirm the token has read-only Contents access.'
)
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# PEFT probes optional quantization backends while creating LoRA modules.
# Colab's preinstalled torchao has caused an import failure even though this
# project does not quantize during training, so remove it before imports.
!pip uninstall -qy torchao
!pip install -q -r requirements-train.txt

# These imports must follow installation in this fresh Colab runtime.
import accelerate  # noqa: E402
import datasets  # noqa: E402
import peft  # noqa: E402
import transformers  # noqa: E402

print('transformers', transformers.__version__)
print('peft', peft.__version__)
print('datasets', datasets.__version__)
print('accelerate', accelerate.__version__)

In [ ]:
import torch

assert torch.cuda.is_available(), 'No CUDA GPU. Select Runtime > Change runtime type > T4 GPU.'
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 1024**3
print('torch', torch.__version__, '| GPU:', gpu.name, '| VRAM:', f'{gpu_gib:.1f} GiB')
assert gpu_gib >= 14, (
    f'{gpu_gib:.1f} GiB is too small for the full run. Select a T4 or a larger GPU.'
)

In [ ]:
# Regenerate the derived dataset with grouped splits. --add-reverse teaches
# both English-to-local and local-to-English without leaking a reversed pair
# or another version of the same verse into a different split.
!python scripts/prepare_all.py data -o data/prepared --add-reverse

In [ ]:
import json
from pathlib import Path

# This streams every prepared row, proves split groups are disjoint, checks
# both Fulfulde script labels, verifies reverse directions, and recomputes
# the SHA-256 values stored in dataset_manifest.json.
!python scripts/validate_prepared.py data/prepared

registry = json.loads(Path('data/languages.json').read_text(encoding='utf-8'))
expected = {code for code, info in registry.items() if info['status'] == 'data_ready'}
manifest = json.loads(Path('data/prepared/dataset_manifest.json').read_text())
assert manifest['reverse_directions_added'] is True
assert set(manifest['target_languages']) == expected
print(f"Validated {len(expected)} language-script targets.")

In [ ]:
# First run two optimizer steps in an ephemeral directory. This catches
# tokenizer, PEFT, CUDA-memory, evaluation, checkpoint, merge, and manifest
# failures before the long Drive-backed job spends hours on the full corpus.
SMOKE_DIR = '/content/jsc_translator_smoke'
if os.path.isdir(SMOKE_DIR):
    shutil.rmtree(SMOKE_DIR)
!python scripts/finetune.py \
  --train data/prepared/train.jsonl \
  --dev data/prepared/dev.jsonl \
  --out {SMOKE_DIR} \
  --epochs 1 --batch 2 --grad-accum 1 --eval-steps 1 \
  --max-steps 2 --max-train-samples 256 --max-dev-samples 64 --resume never
assert os.path.isfile(f'{SMOKE_DIR}/merged/training_manifest.json')
shutil.rmtree(SMOKE_DIR)

# A physical batch of 4 and 16 accumulation steps retain an effective batch
# of 64 while leaving T4 memory headroom for long verses. Unknown language
# tokens are detected directly from the dataset, so no manual list can drift.
!python scripts/finetune.py \
  --train data/prepared/train.jsonl \
  --dev data/prepared/dev.jsonl \
  --out {OUT_DIR} \
  --epochs 1 --batch 4 --grad-accum 16 --eval-steps 500

## After training finishes

`<OUT_DIR>/merged/` (the `OUT_DIR` set two cells above, under `/content/drive/MyDrive/jsc_translator/model_out`) now holds the full fine-tuned model - already in Drive, already safe. Convert it to CTranslate2 int8 for CPU serving:

In [ ]:
CT2_DIR = '/content/drive/MyDrive/jsc_translator/model_ct2'
assert os.path.isfile(f'{OUT_DIR}/merged/training_manifest.json'), 'Training did not finish.'
!ct2-transformers-converter \
  --model {OUT_DIR}/merged \
  --output_dir {CT2_DIR} \
  --quantization int8 \
  --force

Download `model_out/merged/` and `model_ct2/` from Drive to run `app/main.py` locally (`MODEL_DIR`/`TOKENIZER_DIR` env vars), or run `scripts/evaluate.py` against `data/prepared/test.jsonl` first to check quality per language pair before deploying.

Once you're happy with it, flip each trained language's status from `data_ready` to `fine_tuned` in `data/languages.json` and commit - that's what makes `app/main.py` actually expose them via `/translate`.